In [ ]:
!pip -q install minerva-ml

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 219.2/219.2 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.3/216.3 kB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.6/853.6 kB 61.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.2/983.2 kB 80.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.1/284.1 kB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.3/49.3 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 44.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 69.3 MB/s eta 0:00:00


In [ ]:
from pathlib import Path
from PIL import Image
from typing import Any
import cv2
import random
import numpy as np
from minerva.data.readers.png_reader import PNGReader
#from minerva.data.readers.patched_array_reader import NumpyArrayReader
from minerva.data.data_modules.base import MinervaDataModule

from minerva.transforms.transform import _Transform, Identity, Crop, Solarize, PadCrop, Padding, Flip, GrayScale, Solarize, Rotation
from minerva.transforms.random_transform import _RandomSyncedTransform, RandomFlip, RandomGrayScale, RandomSolarize, RandomRotation
from minerva.data.datasets.base import SimpleDataset

#from minerva.models.ssl.byol import BYOL
from torchvision import transforms
import lightning as L
import torch
from torchmetrics import JaccardIndex
import matplotlib.pyplot as plt
import torch.nn as nn
from lightning.pytorch.callbacks import Callback, ModelCheckpoint
from torchvision.models.resnet import resnet50

import copy

In [ ]:
from google.colab import drive
import zipfile
import os

In [ ]:
drive.mount('/content/drive')

Mounted at /content/drive


# fucntions and classes

In [ ]:
from types import SimpleNamespace


config = SimpleNamespace(
    DATA = SimpleNamespace(
        IMG_SIZE = 256,

        MASK_PATCH_SIZE = 32,

        MASK_RATIO = 0.4,

        DATASET_MEAN = [0.485, 0.456, 0.406],
        DATASET_STD = [0.229, 0.224, 0.225]
    ),
    MODEL = SimpleNamespace(
        TYPE = 'ViT',

        VIT = SimpleNamespace(
            PATCH_SIZE = 16
        ),

        SWIN = SimpleNamespace(
            PATCH_SIZE = 16
        )
    )
)

In [ ]:
# --------------------------------------------------------
# SimMIM
# Copyright (c) 2021 Microsoft
# Licensed under The MIT License [see LICENSE for details]
# Written by Ze Liu
# Modified by Zhenda Xie
# --------------------------------------------------------
import torch
import torch.nn as nn
import torch.utils.checkpoint as checkpoint
from timm.models.layers import DropPath, to_2tuple, trunc_normal_

# =====================================================================================================================================================

def window_partition(x, window_size):
    """Partitions input tensor into non-overlapping windows.

    Args:
        x (torch.Tensor): Input tensor of shape (batch_size, height, width, channels).
        window_size (int): Size of the square window.

    Returns:
        torch.Tensor: Partitioned windows of shape (num_windows*batch_size, window_size, window_size, channels).
    """
    B, H, W, C = x.shape
    assert H % window_size == 0 and W % window_size == 0, \
        f"Height ({H}) and Width ({W}) must be divisible by window_size ({window_size})"
    x = x.view(B, H // window_size, window_size, W // window_size, window_size, C)
    windows = x.permute(0, 1, 3, 2, 4, 5).contiguous().view(-1, window_size, window_size, C)
    return windows


def window_reverse(windows, window_size, H, W):
    """Reverses window partitioning to reconstruct the original tensor.

    Args:
        windows (torch.Tensor): Input windows of shape (num_windows*batch_size, window_size, window_size, channels).
        window_size (int): Size of the square window.
        H (int): Height of the original image.
        W (int): Width of the original image.

    Returns:
        torch.Tensor: Reconstructed tensor of shape (batch_size, height, width, channels).
    """
    B = int(windows.shape[0] / (H * W / window_size / window_size))
    x = windows.view(B, H // window_size, W // window_size, window_size, window_size, -1)
    x = x.permute(0, 1, 3, 2, 4, 5).contiguous().view(B, H, W, -1)
    return x


class PatchEmbed(nn.Module):
    """Converts an image into patch embeddings.

    Args:
        img_size (int, optional): Input image size (assumed square). Defaults to 224.
        patch_size (int, optional): Size of each patch. Defaults to 4.
        in_chans (int, optional): Number of input image channels. Defaults to 3.
        embed_dim (int, optional): Number of linear projection output channels. Defaults to 96.
        norm_layer (nn.Module, optional): Normalization layer. Defaults to None.
    """
    def __init__(self, img_size=224, patch_size=4, in_chans=3, embed_dim=96, norm_layer=None):
        super().__init__()
        img_size = to_2tuple(img_size)
        patch_size = to_2tuple(patch_size)
        self.img_size = img_size
        self.patch_size = patch_size
        self.patches_resolution = [img_size[0] // patch_size[0], img_size[1] // patch_size[1]]
        self.num_patches = self.patches_resolution[0] * self.patches_resolution[1]
        self.in_chans = in_chans
        self.embed_dim = embed_dim
        self.proj = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)
        self.norm = norm_layer(embed_dim) if norm_layer is not None else None

    def forward(self, x):
        """Embeds input image into patches and applies optional normalization.

        Args:
            x (torch.Tensor): Input tensor of shape (batch_size, channels, height, width).

        Returns:
            torch.Tensor: Embedded patches of shape (batch_size, num_patches, embed_dim).
        """
        B, C, H, W = x.shape
        assert H == self.img_size[0] and W == self.img_size[1], \
            f"Input image size ({H}*{W}) doesn't match model ({self.img_size[0]}*{self.img_size[1]})."
        x = self.proj(x).flatten(2).transpose(1, 2)
        if self.norm is not None:
            x = self.norm(x)
        return x

    def flops(self):
        """Calculates the number of floating-point operations for patch embedding.

        Returns:
            int: Number of FLOPs.
        """
        Ho, Wo = self.patches_resolution
        flops = Ho * Wo * self.embed_dim * self.in_chans * (self.patch_size[0] * self.patch_size[1])
        if self.norm is not None:
            flops += Ho * Wo * self.embed_dim
        return flops


class Mlp(nn.Module):
    """Multi-layer perceptron with dropout.

    Args:
        in_features (int): Number of input features.
        hidden_features (int, optional): Number of hidden features. Defaults to in_features.
        out_features (int, optional): Number of output features. Defaults to in_features.
        act_layer (nn.Module, optional): Activation function. Defaults to nn.GELU.
        drop (float, optional): Dropout rate. Defaults to 0.0.
    """
    def __init__(self, in_features, hidden_features=None, out_features=None, act_layer=nn.GELU, drop=0.):
        super().__init__()
        out_features = out_features or in_features
        hidden_features = hidden_features or in_features
        self.fc1 = nn.Linear(in_features, hidden_features)
        self.act = act_layer()
        self.fc2 = nn.Linear(hidden_features, out_features)
        self.drop = nn.Dropout(drop)

    def forward(self, x):
        """Applies the MLP to the input tensor.

        Args:
            x (torch.Tensor): Input tensor of shape (batch_size, seq_len, in_features).

        Returns:
            torch.Tensor: Output tensor of shape (batch_size, seq_len, out_features).
        """
        x = self.fc1(x)
        x = self.act(x)
        x = self.drop(x)
        x = self.fc2(x)
        x = self.drop(x)
        return x


class WindowAttention(nn.Module):
    """Window-based multi-head self-attention (W-MSA) with relative position bias.

    Args:
        dim (int): Number of input channels.
        window_size (tuple[int]): Height and width of the window.
        num_heads (int): Number of attention heads.
        qkv_bias (bool, optional): If True, adds learnable bias to query, key, value. Defaults to True.
        qk_scale (float, optional): Override default qk scale of head_dim ** -0.5 if set. Defaults to None.
        attn_drop (float, optional): Dropout ratio of attention weight. Defaults to 0.0.
        proj_drop (float, optional): Dropout ratio of output. Defaults to 0.0.
    """
    def __init__(self, dim, window_size, num_heads, qkv_bias=True, qk_scale=None, attn_drop=0., proj_drop=0.):
        super().__init__()
        self.dim = dim
        self.window_size = window_size
        self.num_heads = num_heads
        head_dim = dim // num_heads
        self.scale = qk_scale or head_dim ** -0.5
        self.relative_position_bias_table = nn.Parameter(
            torch.zeros((2 * window_size[0] - 1) * (2 * window_size[1] - 1), num_heads))
        coords_h = torch.arange(self.window_size[0])
        coords_w = torch.arange(self.window_size[1])
        coords = torch.stack(torch.meshgrid([coords_h, coords_w]))
        coords_flatten = torch.flatten(coords, 1)
        relative_coords = coords_flatten[:, :, None] - coords_flatten[:, None, :]
        relative_coords = relative_coords.permute(1, 2, 0).contiguous()
        relative_coords[:, :, 0] += self.window_size[0] - 1
        relative_coords[:, :, 1] += self.window_size[1] - 1
        relative_coords[:, :, 0] *= 2 * self.window_size[1] - 1
        relative_position_index = relative_coords.sum(-1)
        self.register_buffer("relative_position_index", relative_position_index)
        self.qkv = nn.Linear(dim, dim * 3, bias=qkv_bias)
        self.attn_drop = nn.Dropout(attn_drop)
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(proj_drop)
        trunc_normal_(self.relative_position_bias_table, std=.02)
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, x, mask=None):
        """Computes window-based multi-head self-attention.

        Args:
            x (torch.Tensor): Input tensor of shape (num_windows*batch_size, window_size*window_size, channels).
            mask (torch.Tensor, optional): Attention mask of shape (num_windows, window_size*window_size, window_size*window_size). Defaults to None.

        Returns:
            torch.Tensor: Output tensor of shape (num_windows*batch_size, window_size*window_size, channels).
        """
        B_, N, C = x.shape
        qkv = self.qkv(x).reshape(B_, N, 3, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        q = q * self.scale
        attn = (q @ k.transpose(-2, -1))
        relative_position_bias = self.relative_position_bias_table[self.relative_position_index.view(-1)].view(
            self.window_size[0] * self.window_size[1], self.window_size[0] * self.window_size[1], -1)
        relative_position_bias = relative_position_bias.permute(2, 0, 1).contiguous()
        attn = attn + relative_position_bias.unsqueeze(0)
        if mask is not None:
            nW = mask.shape[0]
            attn = attn.view(B_ // nW, nW, self.num_heads, N, N) + mask.unsqueeze(1).unsqueeze(0)
            attn = attn.view(-1, self.num_heads, N, N)
        attn = self.softmax(attn)
        attn = self.attn_drop(attn)
        x = (attn @ v).transpose(1, 2).reshape(B_, N, C)
        x = self.proj(x)
        x = self.proj_drop(x)
        return x

    def extra_repr(self):
        """Returns a string representation of the module's key attributes.

        Returns:
            str: String containing dimension, window size, and number of heads.
        """
        return f'dim={self.dim}, window_size={self.window_size}, num_heads={self.num_heads}'

    def flops(self, N):
        """Calculates the number of floating-point operations for one window.

        Args:
            N (int): Number of tokens in the window.

        Returns:
            int: Number of FLOPs.
        """
        flops = N * self.dim * 3 * self.dim
        flops += self.num_heads * N * (self.dim // self.num_heads) * N
        flops += self.num_heads * N * N * (self.dim // self.num_heads)
        flops += N * self.dim * self.dim
        return flops


class SwinTransformerBlock(nn.Module):
    """Swin Transformer block with window-based attention and MLP.

    Args:
        dim (int): Number of input channels.
        input_resolution (tuple[int]): Input resolution (height, width).
        num_heads (int): Number of attention heads.
        window_size (int, optional): Window size. Defaults to 7.
        shift_size (int, optional): Shift size for shifted window attention. Defaults to 0.
        mlp_ratio (float, optional): Ratio of MLP hidden dim to embedding dim. Defaults to 4.0.
        qkv_bias (bool, optional): If True, adds learnable bias to query, key, value. Defaults to True.
        qk_scale (float, optional): Override default qk scale of head_dim ** -0.5 if set. Defaults to None.
        drop (float, optional): Dropout rate for MLP. Defaults to 0.0.
        attn_drop (float, optional): Attention dropout rate. Defaults to 0.0.
        drop_path (float, optional): Stochastic depth rate. Defaults to 0.0.
        act_layer (nn.Module, optional): Activation layer. Defaults to nn.GELU.
        norm_layer (nn.Module, optional): Normalization layer. Defaults to nn.LayerNorm.
    """
    def __init__(self, dim, input_resolution, num_heads, window_size=7, shift_size=0,
                 mlp_ratio=4., qkv_bias=True, qk_scale=None, drop=0., attn_drop=0., drop_path=0.,
                 act_layer=nn.GELU, norm_layer=nn.LayerNorm):
        super().__init__()
        self.dim = dim
        self.input_resolution = input_resolution
        self.num_heads = num_heads
        self.window_size = window_size
        self.shift_size = shift_size
        self.mlp_ratio = mlp_ratio
        if min(self.input_resolution) <= self.window_size:
            self.shift_size = 0
            self.window_size = min(self.input_resolution)
        assert 0 <= self.shift_size < self.window_size, "shift_size must be in [0, window_size)"
        self.norm1 = norm_layer(dim)
        self.attn = WindowAttention(
            dim, window_size=to_2tuple(self.window_size), num_heads=num_heads,
            qkv_bias=qkv_bias, qk_scale=qk_scale, attn_drop=attn_drop, proj_drop=drop)
        self.drop_path = DropPath(drop_path) if drop_path > 0. else nn.Identity()
        self.norm2 = norm_layer(dim)
        mlp_hidden_dim = int(dim * mlp_ratio)
        self.mlp = Mlp(in_features=dim, hidden_features=mlp_hidden_dim, act_layer=act_layer, drop=drop)
        if self.shift_size > 0:
            H, W = self.input_resolution
            img_mask = torch.zeros((1, H, W, 1))
            h_slices = (slice(0, -self.window_size), slice(-self.window_size, -self.shift_size), slice(-self.shift_size, None))
            w_slices = (slice(0, -self.window_size), slice(-self.window_size, -self.shift_size), slice(-self.shift_size, None))
            cnt = 0
            for h in h_slices:
                for w in w_slices:
                    img_mask[:, h, w, :] = cnt
                    cnt += 1
            mask_windows = window_partition(img_mask, self.window_size)
            mask_windows = mask_windows.view(-1, self.window_size * self.window_size)
            attn_mask = mask_windows.unsqueeze(1) - mask_windows.unsqueeze(2)
            attn_mask = attn_mask.masked_fill(attn_mask != 0, float(-100.0)).masked_fill(attn_mask == 0, float(0.0))
            self.register_buffer("attn_mask", attn_mask)
        else:
            self.attn_mask = None

    def forward(self, x):
        """Applies the Swin Transformer block to the input tensor.

        Args:
            x (torch.Tensor): Input tensor of shape (batch_size, height*width, channels).

        Returns:
            torch.Tensor: Output tensor of shape (batch_size, height*width, channels).
        """
        H, W = self.input_resolution
        B, L, C = x.shape
        assert L == H * W, "Input feature has wrong size"
        shortcut = x
        x = self.norm1(x)
        x = x.view(B, H, W, C)
        if self.shift_size > 0:
            shifted_x = torch.roll(x, shifts=(-self.shift_size, -self.shift_size), dims=(1, 2))
        else:
            shifted_x = x
        x_windows = window_partition(shifted_x, self.window_size)
        x_windows = x_windows.view(-1, self.window_size * self.window_size, C)
        attn_windows = self.attn(x_windows, mask=self.attn_mask)
        attn_windows = attn_windows.view(-1, self.window_size, self.window_size, C)
        shifted_x = window_reverse(attn_windows, self.window_size, H, W)
        if self.shift_size > 0:
            x = torch.roll(shifted_x, shifts=(self.shift_size, self.shift_size), dims=(1, 2))
        else:
            x = shifted_x
        x = x.view(B, H * W, C)
        x = shortcut + self.drop_path(x)
        x = x + self.drop_path(self.mlp(self.norm2(x)))
        return x

    def extra_repr(self):
        """Returns a string representation of the module's key attributes.

        Returns:
            str: String containing dimension, input resolution, number of heads, window size, shift size, and MLP ratio.
        """
        return f"dim={self.dim}, input_resolution={self.input_resolution}, num_heads={self.num_heads}, " \
               f"window_size={self.window_size}, shift_size={self.shift_size}, mlp_ratio={self.mlp_ratio}"

    def flops(self):
        """Calculates the number of floating-point operations for the block.

        Returns:
            int: Number of FLOPs.
        """
        flops = 0
        H, W = self.input_resolution
        flops += self.dim * H * W
        nW = H * W / self.window_size / self.window_size
        flops += nW * self.attn.flops(self.window_size * self.window_size)
        flops += 2 * H * W * self.dim * self.dim * self.mlp_ratio
        flops += self.dim * H * W
        return flops


class PatchMerging(nn.Module):
    """Merges patches to reduce resolution and increase channels.

    Args:
        input_resolution (tuple[int]): Resolution of input feature (height, width).
        dim (int): Number of input channels.
        norm_layer (nn.Module, optional): Normalization layer. Defaults to nn.LayerNorm.
    """
    def __init__(self, input_resolution, dim, norm_layer=nn.LayerNorm):
        super().__init__()
        self.input_resolution = input_resolution
        self.dim = dim
        self.reduction = nn.Linear(4 * dim, 2 * dim, bias=False)
        self.norm = norm_layer(4 * dim)

    def forward(self, x):
        """Merges patches from the input tensor.

        Args:
            x (torch.Tensor): Input tensor of shape (batch_size, height*width, channels).

        Returns:
            torch.Tensor: Merged tensor of shape (batch_size, height/2*width/2, 2*channels).
        """
        H, W = self.input_resolution
        B, L, C = x.shape
        assert L == H * W, "Input feature has wrong size"
        assert H % 2 == 0 and W % 2 == 0, f"Input size ({H}*{W}) must be even"
        x = x.view(B, H, W, C)
        x0 = x[:, 0::2, 0::2, :]
        x1 = x[:, 1::2, 0::2, :]
        x2 = x[:, 0::2, 1::2, :]
        x3 = x[:, 1::2, 1::2, :]
        x = torch.cat([x0, x1, x2, x3], -1)
        x = x.view(B, -1, 4 * C)
        x = self.norm(x)
        x = self.reduction(x)
        return x

    def extra_repr(self):
        """Returns a string representation of the module's key attributes.

        Returns:
            str: String containing input resolution and dimension.
        """
        return f"input_resolution={self.input_resolution}, dim={self.dim}"

    def flops(self):
        """Calculates the number of floating-point operations for patch merging.

        Returns:
            int: Number of FLOPs.
        """
        H, W = self.input_resolution
        flops = H * W * self.dim
        flops += (H // 2) * (W // 2) * 4 * self.dim * 2 * self.dim
        return flops


class BasicLayer(nn.Module):
    """A basic Swin Transformer layer for one stage.

    Args:
        dim (int): Number of input channels.
        input_resolution (tuple[int]): Input resolution (height, width).
        depth (int): Number of blocks.
        num_heads (int): Number of attention heads.
        window_size (int): Local window size.
        mlp_ratio (float, optional): Ratio of MLP hidden dim to embedding dim. Defaults to 4.0.
        qkv_bias (bool, optional): If True, adds learnable bias to query, key, value. Defaults to True.
        qk_scale (float, optional): Override default qk scale of head_dim ** -0.5 if set. Defaults to None.
        drop (float, optional): Dropout rate. Defaults to 0.0.
        attn_drop (float, optional): Attention dropout rate. Defaults to 0.0.
        drop_path (float or list[float], optional): Stochastic depth rate. Defaults to 0.0.
        norm_layer (nn.Module, optional): Normalization layer. Defaults to nn.LayerNorm.
        downsample (nn.Module, optional): Downsample layer at the end of the layer. Defaults to None.
        use_checkpoint (bool, optional): Whether to use checkpointing to save memory. Defaults to False.
    """
    def __init__(self, dim, input_resolution, depth, num_heads, window_size,
                 mlp_ratio=4., qkv_bias=True, qk_scale=None, drop=0., attn_drop=0.,
                 drop_path=0., norm_layer=nn.LayerNorm, downsample=None, use_checkpoint=False):
        super().__init__()
        self.dim = dim
        self.input_resolution = input_resolution
        self.depth = depth
        self.use_checkpoint = use_checkpoint
        self.blocks = nn.ModuleList([
            SwinTransformerBlock(dim=dim, input_resolution=input_resolution,
                                 num_heads=num_heads, window_size=window_size,
                                 shift_size=0 if (i % 2 == 0) else window_size // 2,
                                 mlp_ratio=mlp_ratio, qkv_bias=qkv_bias, qk_scale=qk_scale,
                                 drop=drop, attn_drop=attn_drop,
                                 drop_path=drop_path[i] if isinstance(drop_path, list) else drop_path,
                                 norm_layer=norm_layer)
            for i in range(depth)])
        self.downsample = downsample(input_resolution, dim=dim, norm_layer=norm_layer) if downsample is not None else None

    def forward(self, x):
        """Applies the layer's transformer blocks and optional downsampling.

        Args:
            x (torch.Tensor): Input tensor of shape (batch_size, height*width, channels).

        Returns:
            torch.Tensor: Output tensor of shape (batch_size, height*width, channels) or reduced resolution if downsampled.
        """
        for blk in self.blocks:
            if self.use_checkpoint:
                x = checkpoint.checkpoint(blk, x)
            else:
                x = blk(x)
        if self.downsample is not None:
            x = self.downsample(x)
        return x

    def extra_repr(self):
        """Returns a string representation of the module's key attributes.

        Returns:
            str: String containing dimension, input resolution, and depth.
        """
        return f"dim={self.dim}, input_resolution={self.input_resolution}, depth={self.depth}"

    def flops(self):
        """Calculates the number of floating-point operations for the layer.

        Returns:
            int: Number of FLOPs.
        """
        flops = 0
        for blk in self.blocks:
            flops += blk.flops()
        if self.downsample is not None:
            flops += self.downsample.flops()
        return flops


class SwinTransformer(nn.Module):
    """Swin Transformer model with hierarchical vision transformer architecture.

    Args:
        img_size (int, optional): Input image size (assumed square). Defaults to 224.
        patch_size (int, optional): Patch size. Defaults to 4.
        in_chans (int, optional): Number of input image channels. Defaults to 3.
        num_classes (int, optional): Number of classes for classification head. Defaults to 1000.
        embed_dim (int, optional): Patch embedding dimension. Defaults to 96.
        depths (list[int], optional): Depth of each Swin Transformer layer. Defaults to [2, 2, 6, 2].
        num_heads (list[int], optional): Number of attention heads in different layers. Defaults to [3, 6, 12, 24].
        window_size (int, optional): Window size. Defaults to 7.
        mlp_ratio (float, optional): Ratio of MLP hidden dim to embedding dim. Defaults to 4.0.
        qkv_bias (bool, optional): If True, adds learnable bias to query, key, value. Defaults to True.
        qk_scale (float, optional): Override default qk scale of head_dim ** -0.5 if set. Defaults to None.
        drop_rate (float, optional): Dropout rate. Defaults to 0.0.
        attn_drop_rate (float, optional): Attention dropout rate. Defaults to 0.0.
        drop_path_rate (float, optional): Stochastic depth rate. Defaults to 0.1.
        norm_layer (nn.Module, optional): Normalization layer. Defaults to nn.LayerNorm.
        ape (bool, optional): If True, adds absolute position embedding. Defaults to False.
        patch_norm (bool, optional): If True, adds normalization after patch embedding. Defaults to True.
        use_checkpoint (bool, optional): Whether to use checkpointing to save memory. Defaults to False.
    """
    def __init__(self, img_size=224, patch_size=4, in_chans=3, num_classes=1000,
                 embed_dim=96, depths=[2, 2, 6, 2], num_heads=[3, 6, 12, 24],
                 window_size=7, mlp_ratio=4., qkv_bias=True, qk_scale=None,
                 drop_rate=0., attn_drop_rate=0., drop_path_rate=0.1,
                 norm_layer=nn.LayerNorm, ape=False, patch_norm=True,
                 use_checkpoint=False, **kwargs):
        super().__init__()
        self.num_classes = num_classes
        self.num_layers = len(depths)
        self.embed_dim = embed_dim
        self.ape = ape
        self.patch_norm = patch_norm
        self.num_features = int(embed_dim * 2 ** (self.num_layers - 1))
        self.mlp_ratio = mlp_ratio
        self.patches_resolution = (img_size // patch_size, img_size // patch_size)
        self.patch_embed = PatchEmbed(
            img_size=img_size, patch_size=patch_size, in_chans=in_chans, embed_dim=embed_dim,
            norm_layer=norm_layer if self.patch_norm else None)
        if self.ape:
            self.absolute_pos_embed = nn.Parameter(torch.zeros(1, self.patch_embed.num_patches, embed_dim))
            trunc_normal_(self.absolute_pos_embed, std=.02)
        self.pos_drop = nn.Dropout(p=drop_rate)
        dpr = [x.item() for x in torch.linspace(0, drop_path_rate, sum(depths))]
        self.layers = nn.ModuleList([
            BasicLayer(
                dim=int(embed_dim * 2 ** i_layer),
                input_resolution=(self.patches_resolution[0] // (2 ** i_layer), self.patches_resolution[1] // (2 ** i_layer)),
                depth=depths[i_layer],
                num_heads=num_heads[i_layer],
                window_size=window_size,
                mlp_ratio=self.mlp_ratio,
                qkv_bias=qkv_bias, qk_scale=qk_scale,
                drop=drop_rate, attn_drop=attn_drop_rate,
                drop_path=dpr[sum(depths[:i_layer]):sum(depths[:i_layer + 1])],
                norm_layer=norm_layer,
                downsample=PatchMerging if (i_layer < self.num_layers - 1) else None,
                use_checkpoint=use_checkpoint)
            for i_layer in range(self.num_layers)])
        self.norm = norm_layer(self.num_features)
        self.avgpool = nn.AdaptiveAvgPool1d(1)
        self.head = nn.Linear(self.num_features, num_classes) if num_classes > 0 else nn.Identity()
        self.apply(self._init_weights)

    def _init_weights(self, m):
        """Initializes weights for Linear and LayerNorm layers.

        Args:
            m (nn.Module): Module to initialize.
        """
        if isinstance(m, nn.Linear):
            trunc_normal_(m.weight, std=.02)
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.LayerNorm):
            nn.init.constant_(m.bias, 0)
            nn.init.constant_(m.weight, 1.0)

    def no_weight_decay(self):
        """Returns parameters that should not have weight decay.

        Returns:
            set: Names of parameters to exclude from weight decay.
        """
        return {'absolute_pos_embed'}

    def no_weight_decay_keywords(self):
        """Returns keywords for parameters that should not have weight decay.

        Returns:
            set: Keywords for parameters to exclude from weight decay.
        """
        return {'relative_position_bias_table'}

    def forward_features(self, x):
        """Processes input through patch embedding and transformer layers.

        Args:
            x (torch.Tensor): Input tensor of shape (batch_size, channels, height, width).

        Returns:
            torch.Tensor: Processed features of shape (batch_size, num_features).
        """
        x = self.patch_embed(x)
        if self.ape:
            x = x + self.absolute_pos_embed
        x = self.pos_drop(x)
        for layer in self.layers:
            x = layer(x)
        x = self.norm(x)
        x = self.avgpool(x.transpose(1, 2))
        x = torch.flatten(x, 1)
        return x

    def forward(self, x):
        """Performs a forward pass through the entire Swin Transformer.

        Args:
            x (torch.Tensor): Input tensor of shape (batch_size, channels, height, width).

        Returns:
            torch.Tensor: Output tensor of shape (batch_size, num_classes).
        """
        x = self.forward_features(x)
        x = self.head(x)
        return x

    def flops(self):
        """Calculates the total number of floating-point operations for the model.

        Returns:
            int: Number of FLOPs.
        """
        flops = self.patch_embed.flops()
        for layer in self.layers:
            flops += layer.flops()
        flops += self.num_features * self.patches_resolution[0] * self.patches_resolution[1] // (2 ** self.num_layers)
        flops += self.num_features * self.num_classes
        return flops

In [ ]:
# --------------------------------------------------------
# SimMIM
# Copyright (c) 2021 Microsoft
# Licensed under The MIT License [see LICENSE for details]
# Written by Ze Liu
# Modified by Zhenda Xie
# --------------------------------------------------------
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from timm.models.layers import DropPath, to_2tuple, trunc_normal_
from functools import partial

# =====================================================================================================================================================

class PatchEmbed(nn.Module):
    """Converts an image into patch embeddings.

    Args:
        img_size (int, optional): Input image size (assumed square). Defaults to 224.
        patch_size (int, optional): Size of each patch. Defaults to 16.
        in_chans (int, optional): Number of input channels. Defaults to 3.
        embed_dim (int, optional): Embedding dimension. Defaults to 768.
    """
    def __init__(self, img_size=224, patch_size=16, in_chans=3, embed_dim=768):
        super().__init__()
        img_size = to_2tuple(img_size)
        patch_size = to_2tuple(patch_size)
        self.img_size = img_size
        self.patch_size = patch_size
        self.patch_shape = (img_size[0] // patch_size[0], img_size[1] // patch_size[1])
        self.num_patches = self.patch_shape[0] * self.patch_shape[1]
        self.proj = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        """Embeds input image into patches and flattens them.

        Args:
            x (torch.Tensor): Input tensor of shape (batch_size, channels, height, width).

        Returns:
            torch.Tensor: Embedded patches of shape (batch_size, num_patches, embed_dim).
        """
        B, C, H, W = x.shape
        assert H == self.img_size[0] and W == self.img_size[1], \
            f"Input image size ({H}*{W}) doesn't match model ({self.img_size[0]}*{self.img_size[1]})."
        x = self.proj(x).flatten(2).transpose(1, 2)
        return x


class Mlp(nn.Module):
    """Multi-layer perceptron with dropout.

    Args:
        in_features (int): Number of input features.
        hidden_features (int, optional): Number of hidden features. Defaults to in_features.
        out_features (int, optional): Number of output features. Defaults to in_features.
        act_layer (nn.Module, optional): Activation function. Defaults to nn.GELU.
        drop (float, optional): Dropout rate. Defaults to 0.0.
    """
    def __init__(self, in_features, hidden_features=None, out_features=None, act_layer=nn.GELU, drop=0.):
        super().__init__()
        out_features = out_features or in_features
        hidden_features = hidden_features or in_features
        self.fc1 = nn.Linear(in_features, hidden_features)
        self.act = act_layer()
        self.fc2 = nn.Linear(hidden_features, out_features)
        self.drop = nn.Dropout(drop)

    def forward(self, x):
        """Applies the MLP to the input tensor.

        Args:
            x (torch.Tensor): Input tensor of shape (batch_size, seq_len, in_features).

        Returns:
            torch.Tensor: Output tensor of shape (batch_size, seq_len, out_features).
        """
        x = self.fc1(x)
        x = self.act(x)
        x = self.fc2(x)
        x = self.drop(x)
        return x


class Attention(nn.Module):
    """Multi-head attention mechanism with optional relative position bias.

    Args:
        dim (int): Input dimension.
        num_heads (int, optional): Number of attention heads. Defaults to 8.
        qkv_bias (bool, optional): If True, adds bias to query and value projections. Defaults to False.
        qk_scale (float, optional): Scaling factor for query-key dot product. Defaults to None.
        attn_drop (float, optional): Dropout rate for attention weights. Defaults to 0.0.
        proj_drop (float, optional): Dropout rate for output projection. Defaults to 0.0.
        window_size (tuple, optional): Window size for relative position bias. Defaults to None.
        attn_head_dim (int, optional): Dimension per attention head. Defaults to None.
    """
    def __init__(self, dim, num_heads=8, qkv_bias=False, qk_scale=None, attn_drop=0.,
                 proj_drop=0., window_size=None, attn_head_dim=None):
        super().__init__()
        self.num_heads = num_heads
        head_dim = dim // num_heads if attn_head_dim is None else attn_head_dim
        all_head_dim = head_dim * self.num_heads
        self.scale = qk_scale or head_dim ** -0.5
        self.qkv = nn.Linear(dim, all_head_dim * 3, bias=False)
        self.q_bias = nn.Parameter(torch.zeros(all_head_dim)) if qkv_bias else None
        self.v_bias = nn.Parameter(torch.zeros(all_head_dim)) if qkv_bias else None
        self.attn_drop = nn.Dropout(attn_drop)
        self.proj = nn.Linear(all_head_dim, dim)
        self.proj_drop = nn.Dropout(proj_drop)

        if window_size:
            self.window_size = window_size
            self.num_relative_distance = (2 * window_size[0] - 1) * (2 * window_size[1] - 1) + 3
            self.relative_position_bias_table = nn.Parameter(
                torch.zeros(self.num_relative_distance, num_heads))
            coords_h = torch.arange(window_size[0])
            coords_w = torch.arange(window_size[1])
            coords = torch.stack(torch.meshgrid([coords_h, coords_w]))
            coords_flatten = torch.flatten(coords, 1)
            relative_coords = coords_flatten[:, :, None] - coords_flatten[:, None, :]
            relative_coords = relative_coords.permute(1, 2, 0).contiguous()
            relative_coords[:, :, 0] += window_size[0] - 1
            relative_coords[:, :, 1] += window_size[1] - 1
            relative_coords[:, :, 0] *= 2 * window_size[1] - 1
            relative_position_index = torch.zeros(size=(window_size[0] * window_size[1] + 1,) * 2,
                                               dtype=relative_coords.dtype)
            relative_position_index[1:, 1:] = relative_coords.sum(-1)
            relative_position_index[0, 0:] = self.num_relative_distance - 3
            relative_position_index[0:, 0] = self.num_relative_distance - 2
            relative_position_index[0, 0] = self.num_relative_distance - 1
            self.register_buffer("relative_position_index", relative_position_index)
        else:
            self.window_size = None
            self.relative_position_bias_table = None
            self.relative_position_index = None

    def forward(self, x, rel_pos_bias=None):
        """Computes multi-head attention on the input tensor.

        Args:
            x (torch.Tensor): Input tensor of shape (batch_size, seq_len, dim).
            rel_pos_bias (torch.Tensor, optional): Relative position bias. Defaults to None.

        Returns:
            torch.Tensor: Output tensor of shape (batch_size, seq_len, dim).
        """
        B, N, C = x.shape
        qkv_bias = torch.cat((self.q_bias, torch.zeros_like(self.v_bias, requires_grad=False), self.v_bias)) \
            if self.q_bias is not None else None
        qkv = F.linear(input=x, weight=self.qkv.weight, bias=qkv_bias)
        qkv = qkv.reshape(B, N, 3, self.num_heads, -1).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        q = q * self.scale
        attn = (q @ k.transpose(-2, -1))

        if self.relative_position_bias_table is not None:
            relative_position_bias = self.relative_position_bias_table[self.relative_position_index.view(-1)].view(
                self.window_size[0] * self.window_size[1] + 1,
                self.window_size[0] * self.window_size[1] + 1, -1)
            relative_position_bias = relative_position_bias.permute(2, 0, 1).contiguous()
            attn = attn + relative_position_bias.unsqueeze(0)

        if rel_pos_bias is not None:
            attn = attn + rel_pos_bias

        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)
        x = (attn @ v).transpose(1, 2).reshape(B, N, -1)
        x = self.proj(x)
        x = self.proj_drop(x)
        return x


class Block(nn.Module):
    """Transformer block with attention and MLP layers.

    Args:
        dim (int): Input dimension.
        num_heads (int): Number of attention heads.
        mlp_ratio (float, optional): Ratio of MLP hidden dim to embedding dim. Defaults to 4.0.
        qkv_bias (bool, optional): If True, adds bias to query, key, value projections. Defaults to False.
        qk_scale (float, optional): Scaling factor for query-key dot product. Defaults to None.
        drop (float, optional): Dropout rate for MLP. Defaults to 0.0.
        attn_drop (float, optional): Dropout rate for attention. Defaults to 0.0.
        drop_path (float, optional): Stochastic depth rate. Defaults to 0.0.
        init_values (float, optional): Initialization value for gamma parameters. Defaults to None.
        act_layer (nn.Module, optional): Activation function for MLP. Defaults to nn.GELU.
        norm_layer (nn.Module, optional): Normalization layer. Defaults to nn.LayerNorm.
        window_size (tuple, optional): Window size for relative position bias. Defaults to None.
        attn_head_dim (int, optional): Dimension per attention head. Defaults to None.
    """
    def __init__(self, dim, num_heads, mlp_ratio=4., qkv_bias=False, qk_scale=None, drop=0., attn_drop=0.,
                 drop_path=0., init_values=None, act_layer=nn.GELU, norm_layer=nn.LayerNorm,
                 window_size=None, attn_head_dim=None):
        super().__init__()
        self.norm1 = norm_layer(dim)
        self.attn = Attention(
            dim, num_heads=num_heads, qkv_bias=qkv_bias, qk_scale=qk_scale,
            attn_drop=attn_drop, proj_drop=drop, window_size=window_size, attn_head_dim=attn_head_dim)
        self.drop_path = DropPath(drop_path) if drop_path > 0. else nn.Identity()
        self.norm2 = norm_layer(dim)
        mlp_hidden_dim = int(dim * mlp_ratio)
        self.mlp = Mlp(in_features=dim, hidden_features=mlp_hidden_dim, act_layer=act_layer, drop=drop)
        self.gamma_1 = nn.Parameter(init_values * torch.ones((dim)), requires_grad=True) if init_values is not None else None
        self.gamma_2 = nn.Parameter(init_values * torch.ones((dim)), requires_grad=True) if init_values is not None else None

    def forward(self, x, rel_pos_bias=None):
        """Applies the transformer block to the input tensor.

        Args:
            x (torch.Tensor): Input tensor of shape (batch_size, seq_len, dim).
            rel_pos_bias (torch.Tensor, optional): Relative position bias. Defaults to None.

        Returns:
            torch.Tensor: Output tensor of shape (batch_size, seq_len, dim).
        """
        if self.gamma_1 is None:
            x = x + self.drop_path(self.attn(self.norm1(x), rel_pos_bias=rel_pos_bias))
            x = x + self.drop_path(self.mlp(self.norm2(x)))
        else:
            x = x + self.drop_path(self.gamma_1 * self.attn(self.norm1(x), rel_pos_bias=rel_pos_bias))
            x = x + self.drop_path(self.gamma_2 * self.mlp(self.norm2(x)))
        return x


class RelativePositionBias(nn.Module):
    """Computes relative position bias for attention.

    Args:
        window_size (tuple): Window size (height, width).
        num_heads (int): Number of attention heads.
    """
    def __init__(self, window_size, num_heads):
        super().__init__()
        self.window_size = window_size
        self.num_relative_distance = (2 * window_size[0] - 1) * (2 * window_size[1] - 1) + 3
        self.relative_position_bias_table = nn.Parameter(torch.zeros(self.num_relative_distance, num_heads))
        coords_h = torch.arange(window_size[0])
        coords_w = torch.arange(window_size[1])
        coords = torch.stack(torch.meshgrid([coords_h, coords_w]))
        coords_flatten = torch.flatten(coords, 1)
        relative_coords = coords_flatten[:, :, None] - coords_flatten[:, None, :]
        relative_coords = relative_coords.permute(1, 2, 0).contiguous()
        relative_coords[:, :, 0] += window_size[0] - 1
        relative_coords[:, :, 1] += window_size[1] - 1
        relative_coords[:, :, 0] *= 2 * window_size[1] - 1
        relative_position_index = torch.zeros(size=(window_size[0] * window_size[1] + 1,) * 2,
                                           dtype=relative_coords.dtype)
        relative_position_index[1:, 1:] = relative_coords.sum(-1)
        relative_position_index[0, 0:] = self.num_relative_distance - 3
        relative_position_index[0:, 0] = self.num_relative_distance - 2
        relative_position_index[0, 0] = self.num_relative_distance - 1
        self.register_buffer("relative_position_index", relative_position_index)

    def forward(self):
        """Computes the relative position bias tensor.

        Returns:
            torch.Tensor: Relative position bias of shape (num_heads, window_size*window_size+1, window_size*window_size+1).
        """
        relative_position_bias = self.relative_position_bias_table[self.relative_position_index.view(-1)].view(
            self.window_size[0] * self.window_size[1] + 1,
            self.window_size[0] * self.window_size[1] + 1, -1)
        return relative_position_bias.permute(2, 0, 1).contiguous()


class VisionTransformer(nn.Module):
    """Vision Transformer with support for patch or hybrid CNN input stage.

    Args:
        img_size (int, optional): Input image size (assumed square). Defaults to 224.
        patch_size (int, optional): Size of each patch. Defaults to 16.
        in_chans (int, optional): Number of input channels. Defaults to 3.
        num_classes (int, optional): Number of output classes. Defaults to 1000.
        embed_dim (int, optional): Embedding dimension. Defaults to 768.
        depth (int, optional): Number of transformer blocks. Defaults to 12.
        num_heads (int, optional): Number of attention heads. Defaults to 12.
        mlp_ratio (float, optional): Ratio of MLP hidden dim to embedding dim. Defaults to 4.0.
        qkv_bias (bool, optional): If True, adds bias to query, key, value projections. Defaults to False.
        qk_scale (float, optional): Scaling factor for query-key dot product. Defaults to None.
        drop_rate (float, optional): Dropout rate. Defaults to 0.0.
        attn_drop_rate (float, optional): Attention dropout rate. Defaults to 0.0.
        drop_path_rate (float, optional): Stochastic depth rate. Defaults to 0.0.
        norm_layer (nn.Module, optional): Normalization layer. Defaults to nn.LayerNorm.
        init_values (float, optional): Initialization value for gamma parameters. Defaults to None.
        use_abs_pos_emb (bool, optional): If True, uses absolute positional embeddings. Defaults to True.
        use_rel_pos_bias (bool, optional): If True, uses relative positional bias. Defaults to False.
        use_shared_rel_pos_bias (bool, optional): If True, shares relative positional bias across blocks. Defaults to False.
        use_mean_pooling (bool, optional): If True, uses mean pooling for final embedding. Defaults to True.
        init_scale (float, optional): Scaling factor for head initialization. Defaults to 0.001.
    """
    def __init__(self, img_size=224, patch_size=16, in_chans=3, num_classes=1000, embed_dim=768, depth=12,
                 num_heads=12, mlp_ratio=4., qkv_bias=False, qk_scale=None, drop_rate=0., attn_drop_rate=0.,
                 drop_path_rate=0., norm_layer=nn.LayerNorm, init_values=None, use_abs_pos_emb=True,
                 use_rel_pos_bias=False, use_shared_rel_pos_bias=False, use_mean_pooling=True, init_scale=0.001):
        super().__init__()
        self.num_classes = num_classes
        self.num_features = self.embed_dim = embed_dim
        self.patch_size = patch_size
        self.in_chans = in_chans
        self.patch_embed = PatchEmbed(img_size=img_size, patch_size=patch_size, in_chans=in_chans, embed_dim=embed_dim)
        num_patches = self.patch_embed.num_patches
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + 1, embed_dim)) if use_abs_pos_emb else None
        self.pos_drop = nn.Dropout(p=drop_rate)
        self.rel_pos_bias = RelativePositionBias(window_size=self.patch_embed.patch_shape, num_heads=num_heads) \
            if use_shared_rel_pos_bias else None
        self.use_rel_pos_bias = use_rel_pos_bias
        dpr = [x.item() for x in torch.linspace(0, drop_path_rate, depth)]
        self.blocks = nn.ModuleList([
            Block(
                dim=embed_dim, num_heads=num_heads, mlp_ratio=mlp_ratio, qkv_bias=qkv_bias, qk_scale=qk_scale,
                drop=drop_rate, attn_drop=attn_drop_rate, drop_path=dpr[i], norm_layer=norm_layer,
                init_values=init_values, window_size=self.patch_embed.patch_shape if use_rel_pos_bias else None)
            for i in range(depth)])
        self.norm = nn.Identity() if use_mean_pooling else norm_layer(embed_dim)
        self.fc_norm = norm_layer(embed_dim) if use_mean_pooling else None
        self.head = nn.Linear(embed_dim, num_classes) if num_classes > 0 else nn.Identity()

        if self.pos_embed is not None:
            self._trunc_normal_(self.pos_embed, std=.02)
        self._trunc_normal_(self.cls_token, std=.02)
        if num_classes > 0:
            self._trunc_normal_(self.head.weight, std=.02)
            self.head.weight.data.mul_(init_scale)
            self.head.bias.data.mul_(init_scale)
        self.apply(self._init_weights)
        self.fix_init_weight()

    def _trunc_normal_(self, tensor, mean=0., std=1.):
        """Applies truncated normal initialization to a tensor.

        Args:
            tensor (torch.Tensor): Tensor to initialize.
            mean (float, optional): Mean of the normal distribution. Defaults to 0.0.
            std (float, optional): Standard deviation of the normal distribution. Defaults to 1.0.
        """
        trunc_normal_(tensor, mean=mean, std=std)

    def _init_weights(self, m):
        """Initializes weights for Linear, LayerNorm, and Conv2d layers.

        Args:
            m (nn.Module): Module to initialize.
        """
        if isinstance(m, nn.Linear):
            self._trunc_normal_(m.weight, std=.02)
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.LayerNorm):
            nn.init.constant_(m.bias, 0)
            nn.init.constant_(m.weight, 1.0)
        elif isinstance(m, nn.Conv2d):
            self._trunc_normal_(m.weight, std=.02)
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)

    def fix_init_weight(self):
        """Rescales weights of attention and MLP layers based on layer depth."""
        for layer_id, layer in enumerate(self.blocks):
            layer.attn.proj.weight.data.div_(math.sqrt(2.0 * (layer_id + 1)))
            layer.mlp.fc2.weight.data.div_(math.sqrt(2.0 * (layer_id + 1)))

    def get_num_layers(self):
        """Returns the number of transformer blocks.

        Returns:
            int: Number of transformer blocks.
        """
        return len(self.blocks)

    def no_weight_decay(self):
        """Returns parameters that should not have weight decay.

        Returns:
            set: Names of parameters to exclude from weight decay.
        """
        return {'pos_embed', 'cls_token'}

    def get_classifier(self):
        """Returns the classifier head.

        Returns:
            nn.Module: The classifier head.
        """
        return self.head

    def reset_classifier(self, num_classes, global_pool=''):
        """Resets the classifier head with a new number of classes.

        Args:
            num_classes (int): Number of output classes.
            global_pool (str, optional): Global pooling method (unused). Defaults to ''.
        """
        self.num_classes = num_classes
        self.head = nn.Linear(self.embed_dim, num_classes) if num_classes > 0 else nn.Identity()

    def forward_features(self, x):
        """Processes input through patch embedding and transformer blocks.

        Args:
            x (torch.Tensor): Input tensor of shape (batch_size, channels, height, width).

        Returns:
            torch.Tensor: Processed features of shape (batch_size, embed_dim).
        """
        x = self.patch_embed(x)
        batch_size, seq_len, _ = x.size()
        cls_tokens = self.cls_token.expand(batch_size, -1, -1)
        x = torch.cat((cls_tokens, x), dim=1)
        if self.pos_embed is not None:
            x = x + self.pos_embed
        x = self.pos_drop(x)
        rel_pos_bias = self.rel_pos_bias() if self.rel_pos_bias is not None else None
        for blk in self.blocks:
            x = blk(x, rel_pos_bias=rel_pos_bias)
        x = self.norm(x)
        if self.fc_norm is not None:
            return self.fc_norm(x[:, 1:, :].mean(1))
        return x[:, 0]

    def forward(self, x):
        """Performs a forward pass through the entire Vision Transformer.

        Args:
            x (torch.Tensor): Input tensor of shape (batch_size, channels, height, width).

        Returns:
            torch.Tensor: Output tensor of shape (batch_size, num_classes).
        """
        x = self.forward_features(x)
        x = self.head(x)
        return x

In [ ]:
# --------------------------------------------------------
# SimMIM in PyTorch Lightning
# Adapted from original SimMIM (Copyright (c) 2021 Microsoft)
# Licensed under The MIT License [see LICENSE for details]
# Original written by Zhenda Xie
# Adapted to PyTorch Lightning while staying as faithful as possible to the original
# --------------------------------------------------------
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms.functional as F_v2

from torchvision.transforms.v2 import Compose, ToImage, ToDtype, RandomResizedCrop, RandomHorizontalFlip, RandomApply, ColorJitter, Normalize
from typing import Optional, List
from timm.models.layers import trunc_normal_
#from minerva.models.nets.image.swinori import SwinTransformer
#from minerva.models.nets.image.vit_timm import VisionTransformer

import os
import numpy as np
import lightning as L
import matplotlib.pyplot as plt

# =====================================================================================================================================================
class SimMIMTransform:
    """Transformation pipeline for SimMIM pretraining, including image augmentation and mask generation.

    Args:
        config: Configuration object containing dataset and model parameters.
    """
    def __init__(self, config):
        self.transform_img = Compose([
            ToImage(),
            ToDtype(torch.float32, scale=True),
            RandomResizedCrop(
                size=config.DATA.IMG_SIZE,
                scale=(0.2, 1.0),
                ratio=(3. / 4., 4. / 3.),
                antialias=True
            ),
            RandomHorizontalFlip(p=0.5),
            RandomApply(
                [ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1)],
                p=0.5
            ),
            Normalize(mean=config.DATA.DATASET_MEAN, std=config.DATA.DATASET_STD)
        ])

        if config.MODEL.TYPE == 'Swin':
            model_patch_size = config.MODEL.SWIN.PATCH_SIZE
        elif config.MODEL.TYPE == 'ViT':
            model_patch_size = config.MODEL.VIT.PATCH_SIZE
        else:
            raise NotImplementedError(f"Unknown model type: {config.MODEL.TYPE}")

        self.mask_generator = MaskGenerator(
            input_size=config.DATA.IMG_SIZE,
            mask_patch_size=config.DATA.MASK_PATCH_SIZE,
            model_patch_size=model_patch_size,
            mask_ratio=config.DATA.MASK_RATIO
        )

    def __call__(self, img):
        """Applies image transformations and generates a random mask.

        Args:
            img: Input image to transform.

        Returns:
            tuple: Transformed image (torch.Tensor) and mask (torch.Tensor).
        """
        img = self.transform_img(img)
        mask = torch.from_numpy(self.mask_generator()).float()
        return img, mask

class MaskGenerator:
    """Generates random masks for SimMIM pretraining.

    Args:
        input_size (int): Size of the input image (default: 192).
        mask_patch_size (int): Size of the mask patches (default: 32).
        model_patch_size (int): Size of the model patches (default: 4).
        mask_ratio (float): Ratio of patches to mask (default: 0.6).
    """
    def __init__(self, input_size=192, mask_patch_size=32, model_patch_size=4, mask_ratio=0.6):
        self.input_size = input_size
        self.mask_patch_size = mask_patch_size
        self.model_patch_size = model_patch_size
        self.mask_ratio = mask_ratio

        assert self.input_size % self.mask_patch_size == 0, \
            f"input_size ({self.input_size}) must be divisible by mask_patch_size ({self.mask_patch_size})"
        assert self.mask_patch_size % self.model_patch_size == 0, \
            f"mask_patch_size ({self.mask_patch_size}) must be divisible by model_patch_size ({self.model_patch_size})"

        self.rand_size = self.input_size // self.mask_patch_size
        self.scale = self.mask_patch_size // self.model_patch_size
        self.token_count = self.rand_size ** 2
        self.mask_count = int(np.ceil(self.token_count * self.mask_ratio))

    def __call__(self):
        """Generates a random binary mask for SimMIM.

        Returns:
            numpy.ndarray: Binary mask of shape (rand_size * scale, rand_size * scale).
        """
        mask_idx = np.random.permutation(self.token_count)[:self.mask_count]
        mask = np.zeros(self.token_count, dtype=int)
        mask[mask_idx] = 1
        mask = mask.reshape((self.rand_size, self.rand_size))
        mask = mask.repeat(self.scale, axis=0).repeat(self.scale, axis=1)
        return mask

class VisionTransformerForSimMIM(VisionTransformer):
    """Vision Transformer adapted for SimMIM pretraining.

    Args:
        **kwargs: Keyword arguments for the parent VisionTransformer class.
    """
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        assert self.num_classes == 0

        self.mask_token = nn.Parameter(torch.zeros(1, 1, self.embed_dim))
        self._trunc_normal_(self.mask_token, std=.02)

    def _trunc_normal_(self, tensor, mean=0., std=1.):
        """Initialize tensor values from a truncated normal distribution."""
        trunc_normal_(tensor, mean=mean, std=std, a=-std, b=std)

    # def forward(self, x, mask):
    #     """
    #     Forward pass for SimMIM pretraining.

    #     Args:
    #         x (torch.Tensor): Input images of shape (B, C, H, W).
    #         mask (torch.Tensor): Binary mask of shape (B, H/patch_size, W/patch_size),
    #             where 1 indicates a masked patch.

    #     Returns:
    #         torch.Tensor: Reconstructed feature map of shape (B, embed_dim, H', W'),
    #         where H' and W' depend on the number of patches.
    #     """
    #     x = self.patch_embed(x)
    #     B, L, _ = x.shape

    #     mask_token = self.mask_token.expand(B, L, -1)
    #     w = mask.flatten(1).unsqueeze(-1).type_as(mask_token)
    #     x = x * (1 - w) + mask_token * w
    #     cls_tokens = self.cls_token.expand(B, -1, -1)
    #     x = torch.cat((cls_tokens, x), dim=1)

    #     if self.pos_embed is not None:
    #         x = x + self.pos_embed
    #     x = self.pos_drop(x)
    #     rel_pos_bias = self.rel_pos_bias() if self.rel_pos_bias is not None else None
    #     for blk in self.blocks:
    #         x = blk(x, rel_pos_bias=rel_pos_bias)
    #     x = self.norm(x)

    #     x = x[:, 1:]
    #     B, L, C = x.shape
    #     H = W = int(L ** 0.5)
    #     x = x.permute(0, 2, 1).reshape(B, C, H, W)
    #     return x

    def forward(self, x, mask=None):
        """
        Forward pass for SimMIM pretraining.

        Args:
            x (torch.Tensor): Input images of shape (B, C, H, W).
            mask (torch.Tensor): Binary mask of shape (B, H/patch_size, W/patch_size),
                where 1 indicates a masked patch.

        Returns:
            torch.Tensor: Reconstructed feature map of shape (B, embed_dim, H', W'),
            where H' and W' depend on the number of patches.
        """
        x = self.patch_embed(x)
        B, L, _ = x.shape

        if mask is not None:
            expected_h = expected_w = int(L ** 0.5)
            if len(mask.shape) == 3:
                mask = F_v2.resize(mask, size=(expected_h, expected_w), interpolation=F_v2.InterpolationMode.NEAREST)
            elif len(mask.shape) == 2:
                mask = mask.view(B, int(mask.shape[1] ** 0.5), int(mask.shape[1] ** 0.5))
                mask = F_v2.resize(mask, size=(expected_h, expected_w), interpolation=F_v2.InterpolationMode.NEAREST)
            mask = mask.flatten(1)
            mask_token = self.mask_token.expand(B, L, -1)
            w = mask.unsqueeze(-1).type_as(mask_token)
            x = x * (1 - w) + mask_token * w

        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat((cls_tokens, x), dim=1)

        if self.pos_embed is not None:
            x = x + self.pos_embed
        x = self.pos_drop(x)

        rel_pos_bias = self.rel_pos_bias() if self.rel_pos_bias is not None else None
        for blk in self.blocks:
            x = blk(x, rel_pos_bias=rel_pos_bias)
        x = self.norm(x)

        x = x[:, 1:]
        B, L, C = x.shape
        H = W = int(L ** 0.5)
        x = x.permute(0, 2, 1).reshape(B, C, H, W)
        return x

class SwinTransformerForSimMIM(SwinTransformer):
    """Swin Transformer adapted for SimMIM pretraining.

    Args:
        **kwargs: Keyword arguments for the parent SwinTransformer class.
    """
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        assert self.num_classes == 0
        self.in_chans = kwargs.get('in_chans', 3)
        self.patch_size = kwargs.get('patch_size', 4)
        self.num_features = int(self.embed_dim * 2 ** (self.num_layers - 1))
        self.mask_token = nn.Parameter(torch.zeros(1, 1, self.embed_dim))
        trunc_normal_(self.mask_token, mean=0., std=.02)

    def forward(self, x, mask=None):
        """Forward pass through the Swin Transformer for SimMIM.

        Args:
            x (torch.Tensor): Input tensor of shape (batch_size, channels, height, width).
            mask (torch.Tensor, optional): Mask tensor of shape (batch_size, height/patch_size, width/patch_size) or (batch_size, num_patches). Defaults to None.

        Returns:
            torch.Tensor: Output tensor of shape (batch_size, embed_dim, height/patch_size, width/patch_size).
        """
        expected_h = expected_w = self.patches_resolution[0]
        x = self.patch_embed(x)

        if mask is not None:
            if len(mask.shape) == 1:
                B = x.shape[0]
                mask = mask.view(B, 1, 1).expand(B, expected_h, expected_w)
            elif len(mask.shape) == 2:
                B = x.shape[0]
                mask = mask.view(B, expected_h, expected_w)
            assert len(mask.shape) == 3 and mask.shape[1:] == (expected_h, expected_w), \
                f"Expected mask shape [B, {expected_h}, {expected_w}], got {mask.shape}"

            B, L, _ = x.shape
            mask_tokens = self.mask_token.expand(B, L, -1)
            w = mask.flatten(1).unsqueeze(-1).type_as(mask_tokens)
            x = x * (1. - w) + mask_tokens * w

        if self.ape:
            x = x + self.absolute_pos_embed
        x = self.pos_drop(x)

        for layer in self.layers:
            x = layer(x)

        x = self.norm(x)
        x = x.transpose(1, 2)
        B, C, L = x.shape
        H = W = int(L ** 0.5)
        x = x.reshape(B, C, H, W)
        return x

    @torch.jit.ignore
    def no_weight_decay(self):
        """Returns parameters to exclude from weight decay.

        Returns:
            set: Set of parameter names to exclude from weight decay.
        """
        return super().no_weight_decay() | {'mask_token'}

class SimMIMDecoder(nn.Module):
    """Decoder for SimMIM to reconstruct images from encoder features.

    Args:
        in_channels (int): Number of input channels from the encoder's output.
        encoder_stride (int): Stride of the encoder's patch embedding (e.g., 16 for a
            192x192 image downsampled to 12x12).
    """
    def __init__(self, in_channels: int, encoder_stride: int):
        super().__init__()
        self.decoder = nn.Sequential(
            nn.Conv2d(
                in_channels=in_channels,
                out_channels=encoder_stride ** 2 * 3,
                kernel_size=1
            ),
            nn.PixelShuffle(encoder_stride)
        )

        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.xavier_normal_(m.weight, gain=1.0)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0.0)

    def forward(self, x):
        """Reconstructs the image from the encoder's feature map.

        Args:
            x (torch.Tensor): Encoder output tensor of shape (batch_size, in_channels,
                height/encoder_stride, width/encoder_stride).

        Returns:
            torch.Tensor: Reconstructed image tensor of shape (batch_size, 3, height, width).
        """
        return self.decoder(x)

class SimMIM(L.LightningModule):
    """SimMIM model for self-supervised masked image modeling.

    Args:
        encoder (Optional[nn.Module]): Backbone encoder (e.g., VisionTransformerForSimMIM or SwinTransformerForSimMIM). Defaults to None.
        encoder_stride (int): Stride of the encoder's patch embedding. Defaults to 16.
        decoder (Optional[nn.Module]): Decoder module for reconstruction. Defaults to None (uses SimMIMDecoder).
        learning_rate (float): Learning rate for the optimizer. Defaults to 2e-3.
        weight_decay (float): Weight decay for the optimizer. Defaults to 0.05.
        dataset_mean (Optional[List[float]]): Mean values for dataset normalization. Defaults to None (skips denormalization).
        dataset_std (Optional[List[float]]): Standard deviation values for dataset normalization. Defaults to None (skips denormalization).
        debug_path (Optional[str]): Directory for saving visualizations. Defaults to None (uses LOG_BASE_DIR).
        save_images (bool): Whether to save reconstruction visualizations. Defaults to False.
    """
    def __init__(
        self,
        encoder: Optional[nn.Module] = None,
        decoder: Optional[nn.Module] = None,
        encoder_stride: int = 32,
        learning_rate: float = 8e-4,
        weight_decay: float = 0.05,
        dataset_mean: Optional[List[float]] = None,
        dataset_std: Optional[List[float]] = None,
        debug_path: Optional[str] = None,
        save_images: bool = False
    ):
        super().__init__()
        self.save_hyperparameters(ignore=['encoder', 'decoder'])
        self.encoder = encoder
        self.encoder_stride = encoder_stride
        self.decoder = decoder if decoder is not None else SimMIMDecoder(
            in_channels=self.encoder.num_features,
            encoder_stride=self.encoder_stride
        )
        self.in_chans = self.encoder.in_chans
        self.patch_size = self.encoder.patch_size
        self.learning_rate = learning_rate
        self.weight_decay = weight_decay
        self.dataset_mean = dataset_mean
        self.dataset_std = dataset_std
        self.debug_path = debug_path
        self.save_images = save_images

    def forward(self, x, mask):
        """Computes the reconstruction loss for masked image modeling.

        Args:
            x (torch.Tensor): Input image tensor of shape (batch_size, channels, height, width).
            mask (torch.Tensor): Mask tensor of shape (batch_size, height/patch_size, width/patch_size).

        Returns:
            torch.Tensor: L1 reconstruction loss (scalar).
        """
        z = self.encoder(x, mask)
        x_rec = self.decoder(z)
        mask = mask.repeat_interleave(self.patch_size, 1).repeat_interleave(self.patch_size, 2).unsqueeze(1).contiguous()
        loss_recon = F.l1_loss(x, x_rec, reduction='none')
        loss = (loss_recon * mask).sum() / (mask.sum() + 1e-5) / self.in_chans
        return loss

    def _compute_psnr(self, x, x_rec):
        """Calculates PSNR between original and reconstructed images.

        Args:
            x (torch.Tensor): Original image tensor of shape (batch_size, channels, height, width).
            x_rec (torch.Tensor): Reconstructed image tensor of shape (batch_size, channels, height, width).

        Returns:
            torch.Tensor: PSNR value (scalar).
        """
        mse = F.mse_loss(x, x_rec, reduction='mean')
        max_i = 1.0
        psnr = 10 * torch.log10(max_i**2 / (mse + 1e-8))
        return psnr

    def training_step(self, batch, batch_idx):
        """Performs a training step and logs the loss.

        Args:
            batch: Tuple of (images, masks) from the dataloader.
            batch_idx: Index of the current batch.

        Returns:
            torch.Tensor: Training loss (scalar).
        """
        x, mask = batch
        loss = self(x, mask)
        self.log('train_loss', loss, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        """Performs a validation step and logs the loss.

        Args:
            batch: Tuple of (images, masks) from the dataloader.
            batch_idx: Index of the current batch.

        Returns:
            torch.Tensor: Validation loss (scalar).
        """
        x, mask = batch
        loss = self(x, mask)
        self.log('val_loss', loss, prog_bar=True)
        return loss

    def test_step(self, batch, batch_idx):
        """Performs a test step and logs the loss.

        Args:
            batch: Tuple of (images, masks) from the dataloader.
            batch_idx: Index of the current batch.

        Returns:
            torch.Tensor: Test loss (scalar).
        """
        x, mask = batch
        loss = self(x, mask)
        self.log('test_loss', loss, prog_bar=True)
        return loss

    def configure_optimizers(self):
        """Configures the optimizer and learning rate scheduler.

        Returns:
            dict: Dictionary containing optimizer, scheduler, and gradient clipping configuration.
        """
        optimizer = torch.optim.AdamW(
            self.parameters(),
            lr=self.learning_rate,
            weight_decay=self.weight_decay
        )
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)
        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "interval": "epoch"
            },
            "gradient_clip_val": 1.0
        }

    def on_training_epoch_end(self):
        """Logs PSNR for full and hybrid reconstructions and optionally saves visualizations at the end of a training epoch."""
        train_dataloader = self.trainer.datamodule.train_dataloader()
        batch = next(iter(train_dataloader))
        x, mask = batch
        x = x.to(self.device)
        mask = mask.to(self.device)

        with torch.no_grad():
            z = self.encoder(x, mask)
            x_rec = self.decoder(z)
            psnr_full = self._compute_psnr(x, x_rec)
            self.log('train_psnr_full', psnr_full, prog_bar=True)

            # Compute PSNR for hybrid image
            mask_resized = F_v2.resize(mask, size=x.shape[2:], interpolation=F_v2.InterpolationMode.NEAREST).unsqueeze(1)
            hybrid = x * (1 - mask_resized) + x_rec * mask_resized
            psnr_hybrid = self._compute_psnr(x, hybrid)
            self.log('train_psnr_hybrid', psnr_hybrid, prog_bar=True)

        if not self.save_images:
            return

        # Denormalize for visualization if normalization parameters are provided
        if self.dataset_mean is not None and self.dataset_std is not None:
            std = torch.tensor(self.dataset_std).to(self.device).view(1, -1, 1, 1)
            mean = torch.tensor(self.dataset_mean).to(self.device).view(1, -1, 1, 1)
            x_denorm = x * std + mean
            x_rec_denorm = x_rec * std + mean
        else:
            x_denorm = x
            x_rec_denorm = x_rec

        log_dir = self.debug_path
        save_path = os.path.join(log_dir, f"epoch_{self.current_epoch}_train_reconstruction.png")
        self.visualize_reconstruction(x_denorm[0], x_rec_denorm[0], mask[0], title=f"Train Epoch {self.current_epoch}", save_path=save_path)

    def on_validation_epoch_end(self):
        """Logs PSNR and optionally saves reconstruction visualizations at the end of a validation epoch."""
        val_dataloader = self.trainer.datamodule.val_dataloader()
        batch = next(iter(val_dataloader))
        x, mask = batch
        x = x.to(self.device)
        mask = mask.to(self.device)

        with torch.no_grad():
            z = self.encoder(x, mask)
            x_rec = self.decoder(z)
            psnr_full = self._compute_psnr(x, x_rec)
            self.log('val_psnr_full', psnr_full, prog_bar=True)

            # Compute PSNR for hybrid image
            mask_resized = F_v2.resize(mask, size=x.shape[2:], interpolation=F_v2.InterpolationMode.NEAREST).unsqueeze(1)
            hybrid = x * (1 - mask_resized) + x_rec * mask_resized
            psnr_hybrid = self._compute_psnr(x, hybrid)
            self.log('val_psnr_hybrid', psnr_hybrid, prog_bar=True)

        if not self.save_images:
            return

        # Denormalize for visualization
        if self.dataset_mean is not None and self.dataset_std is not None:
            std = torch.tensor(self.dataset_std).to(self.device).view(1, -1, 1, 1)
            mean = torch.tensor(self.dataset_mean).to(self.device).view(1, -1, 1, 1)
            x_denorm = x * std + mean
            x_rec_denorm = x_rec * std + mean
        else:
            x_denorm = x
            x_rec_denorm = x_rec

        log_dir = self.debug_path
        save_path = os.path.join(log_dir, f"epoch_{self.current_epoch}_val_reconstruction.png")
        self.visualize_reconstruction(x_denorm[0], x_rec_denorm[0], mask[0], title=f"Validation Epoch {self.current_epoch}", save_path=save_path)

    def visualize_reconstruction(self, x, x_rec, mask, title="Reconstruction", save_path=None, debug_path=None):
        """Visualizes original, masked, reconstructed, and hybrid images.

        Args:
            x (torch.Tensor): Original image tensor of shape (channels, height, width).
            x_rec (torch.Tensor): Reconstructed image tensor of shape (channels, height, width).
            mask (torch.Tensor): Mask tensor of shape (height/patch_size, width/patch_size).
            title (str): Title for the visualization plot. Defaults to "Reconstruction".
            save_path (str, optional): Path to save the visualization image. Defaults to None.
            debug_path (str, optional): Ignored parameter for backward compatibility. Defaults to None.
        """
        x = x.permute(1, 2, 0).cpu().clamp(0, 1)
        x_rec = x_rec.permute(1, 2, 0).cpu().clamp(0, 1)
        mask_resized = F_v2.resize(mask.unsqueeze(0), size=x.shape[:2], interpolation=F_v2.InterpolationMode.NEAREST).squeeze(0).cpu()

        # Hybrid reconstruction
        hybrid = x * (1 - mask_resized.unsqueeze(-1)) + x_rec * mask_resized.unsqueeze(-1)

        _, axes = plt.subplots(1, 4, figsize=(20, 5))
        axes[0].imshow(x.numpy())
        axes[0].set_title("Original Image")
        axes[1].imshow(mask_resized.numpy(), cmap="gray")
        axes[1].set_title("Mask")
        axes[2].imshow(x_rec.numpy())
        axes[2].set_title("Full Reconstruction")
        axes[3].imshow(hybrid.numpy())
        axes[3].set_title("Hybrid (Original + Reconstructed Patches)")
        plt.suptitle(title)

        if save_path:
            os.makedirs(os.path.dirname(save_path), exist_ok=True)
            plt.savefig(save_path, bbox_inches='tight')
        plt.close()

    @torch.jit.ignore
    def no_weight_decay(self):
        """Returns parameters to exclude from weight decay.

        Returns:
            set: Set of parameter names to exclude from weight decay.
        """
        if hasattr(self.encoder, 'no_weight_decay'):
            return {'encoder.' + i for i in self.encoder.no_weight_decay()}
        return {}

    @torch.jit.ignore
    def no_weight_decay_keywords(self):
        """Returns parameter keywords to exclude from weight decay.

        Returns:
            set: Set of parameter keywords to exclude from weight decay.
        """
        if hasattr(self.encoder, 'no_weight_decay_keywords'):
            return {'encoder.' + i for i in self.encoder.no_weight_decay_keywords()}
        return {}

# unzip dataset

In [ ]:
# path to the main dataset
!unzip -q '/content/drive/My Drive/spectogram_rgb.zip' -d'/content/'

In [ ]:
train_data_reader = PNGReader(path=Path('/content/spectogram_rgb'))

In [ ]:
# path to other datasets, like single dataset or minus 1 dataset
!unzip -q '/content/drive/My Drive/other_datasets.zip' -d'/content/'

In [ ]:
train_data_reader_other_dataset = PNGReader(path=Path('/content/other_datasets'))

# dataset + reader

In [ ]:
dataset_0 = SimpleDataset(
        readers=[train_data_reader],
        transforms=[SimMIMTransform(config)],
        return_single = True
    )

In [ ]:
dataset_1 = SimpleDataset(
        readers=[train_data_reader_other_dataset],
        transforms=[SimMIMTransform(config)],
        return_single = True
  )

# dataset config

In [ ]:
train_dataset_config_0 = dataset_0

In [ ]:
train_dataset_config_1 = dataset_1

# data module

In [ ]:
data_module_0 = MinervaDataModule(
        train_dataset=train_dataset_config_0,
        batch_size=128,
        num_workers=8,
        name="Spectogram Dataset"
    )

In [ ]:
data_module_1 = MinervaDataModule(
        train_dataset=train_dataset_config_1,
        batch_size=128,
        num_workers=8,
        name="Spectogram Dataset"
    )

# modelos

In [ ]:
model_0 = SimMIM(
    encoder = VisionTransformerForSimMIM(img_size=256, num_classes=0),
    encoder_stride=16
)

In [ ]:
model_1 = SimMIM(
    encoder = VisionTransformerForSimMIM(img_size=256, num_classes=0),
    encoder_stride=16
)

In [ ]:
checkpoint_callback_0 = ModelCheckpoint(
    every_n_epochs=5,
    save_top_k=-1,
    # path to save checkpoint folder
    dirpath="/content/drive/MyDrive/checkpoints/",
    filename="simmim_full_epoch{epoch:02d}"
  )

In [ ]:
checkpoint_callback_1 = ModelCheckpoint(
    every_n_epochs=5,
    save_top_k=-1,
    # path to save checkpoint folder
    dirpath="/content/drive/MyDrive/iniciacao_cientifica/checkpoints/",
    filename="simmim_no_icarus_epoch{epoch:02d}"
  )

# trainer

In [ ]:
trainer_0 = L.Trainer(
        max_epochs=30,
        accelerator="gpu",
        devices=1,
        logger=False,
        enable_checkpointing=True,
        callbacks=[checkpoint_callback_0]
    )

In [ ]:
trainer_1 = L.Trainer(
        max_epochs=30,
        accelerator="gpu",
        devices=1,
        logger=False,
        enable_checkpointing=True,
        callbacks=[checkpoint_callback_1]
    )

# training

In [ ]:
trainer_0.fit(model_0, data_module_0)

In [ ]:
trainer_1.fit(model_1, data_module_1)